# Lakelink Fiber PRI: v4 candidate writers (@data-engineer)

This is the versioned writer for the v4 shadow-build objects that @data-engineer owns (task P37-DE-V4-WRITERS-CHECKIN). All write targets are `*_v4_candidate` objects in `cdm_tmforum.provable_resilience_intelligence`. It never writes a v3 FROZEN object. v3 inputs are read only through their FREEZE-v3 pins (`VERSION AS OF`).

| Cell | Object | Rule / task |
| --- | --- | --- |
| 3 | `physical_resource_span_v4_candidate` | B1: BUG-DP-068 BB to BBS segmentation, plus the BUG-QA-075 planar chord split at equal haversine arc length |
| 4 | `physical_resource_conduit_v4_candidate` | A1: BUG-QA-066 geodesic length; P35 coverage class; B2: CDT-BBR-* route conduits |
| 5 | `hazard_span_intersection_v4_candidate` | BUG-QA-065 line distance and BUG-QA-064 rounding to 4 dp, from span@11 |
| 6 | `resource_alarm_v4_candidate`, `resource_performance_measure_v4_candidate` | B5: BUG-QA-072 hash-rank round-robin remap |
| 7 | `mv_mrc_protected_v4_candidate`, `hazard_proximity_count_v4_candidate` | Metric view DDL, verbatim |
| 8 | Gates | Recompute vs live, 0 diffs expected |

**Modes.** The `write_enabled` widget defaults to `false`. In that mode (dry run), every object is recomputed into a temp view and diffed against the live table, and nothing is written. With `true`, tables are overwritten with `CREATE TABLE IF NOT EXISTS` followed by `INSERT OVERWRITE`, which keeps column comments. Metric views are re-created. Writes are only allowed inside an approved v4 write window (DEC-PRI-V4-SHADOW-BUILD). During the freeze, run in dry-run mode only.

**Pins.** `physical_resource_span@20`, `physical_resource_span@11` (HSI only), `physical_resource_conduit@5`, `hazard_span_intersection@22`, `geographic_hazard_event@34`, `resource_alarm@12`, `resource_performance_measure@8`. Build timestamps are fixed constants, so a rerun reproduces the tables exactly.

**Known limit.** The HSI candidate is still keyed on v3 `BB-*` route ids (18,495 rows). Its rebuild on the BBS ids is BUG-QA-073 (@data-planner) and is out of scope here.

In [0]:
import math
import re

import numpy as np
import pandas as pd

dbutils.widgets.dropdown("write_enabled", "false", ["false", "true"], "Write v4 candidate objects")
WRITE_ENABLED = dbutils.widgets.get("write_enabled") == "true"

S = "cdm_tmforum.provable_resilience_intelligence"
SPAN_V4 = f"{S}.physical_resource_span_v4_candidate"
CONDUIT_V4 = f"{S}.physical_resource_conduit_v4_candidate"
HSI_V4 = f"{S}.hazard_span_intersection_v4_candidate"
ALARM_V4 = f"{S}.resource_alarm_v4_candidate"
PERF_V4 = f"{S}.resource_performance_measure_v4_candidate"
MV_MRC_V4 = f"{S}.mv_mrc_protected_v4_candidate"
MV_HPC_V4 = f"{S}.hazard_proximity_count_v4_candidate"

# FREEZE-v3 read pins (reads only)
PIN_SPAN = 20          # physical_resource_span@20 (identical data to @11; 12-20 are comment/property changes)
PIN_SPAN_HSI = 11      # physical_resource_span@11, the pin the HSI candidate was measured on
PIN_CONDUIT = 5        # physical_resource_conduit@5
PIN_HSI = 22           # hazard_span_intersection@22
PIN_HAZARD = 34        # geographic_hazard_event@34
PIN_ALARM = 12         # resource_alarm@12
PIN_PERF = 8           # resource_performance_measure@8

# Fixed build constants, so a rerun reproduces the live candidates exactly
B2_BUILD_TS = "2026-10-25 21:09:47.991845"   # created_at of the 92 CDT-BBR-* conduits
HSI_BUILD_TS = "2026-10-25 13:14:35.27262"   # computed_at of every HSI candidate row
HSI_BATCH_ID = "v4cand-bugqa065-064-20261025"
MAX_SEG_KM = 25.0      # BBS segment cap (B1)
ARC_STEPS = 20000      # arc-length sampling steps along each planar route chord (BUG-QA-075)
DENSIFY_KM = 5.0       # great-circle densification step for HSI span lines (BUG-QA-065)
R_KM = 6371.0088       # mean Earth radius used by every haversine in this notebook

GATES = []


def assert_v4_target(name: str) -> None:
    """Runtime companion to tests/test_v4_frozen_write_guard.py."""
    assert name.startswith(S + ".") and name.endswith("_v4_candidate"), f"refusing non-v4 target {name}"


for _t in (SPAN_V4, CONDUIT_V4, HSI_V4, ALARM_V4, PERF_V4, MV_MRC_V4, MV_HPC_V4):
    assert_v4_target(_t)


def gate(name: str, ok: bool, detail) -> None:
    GATES.append({"gate": name, "pass": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + ": " + str(detail))


def diff_vs_live(view: str, target: str, key: str, label: str) -> None:
    """Dry-run gate: recomputed temp view vs live table, full outer join on key and xxhash64 over all columns."""
    cols = spark.table(target).columns
    assert spark.table(view).columns == cols, f"{label}: column order/name drift vs {target}"
    h = lambda a: "xxhash64(" + ", ".join(f"{a}.`{c}`" for c in cols) + ")"
    r = spark.sql(f"""
      SELECT count(r.{key}) AS n_rec, count(t.{key}) AS n_live,
             coalesce(sum(int(r.{key} IS NULL)), 0) AS only_live,
             coalesce(sum(int(t.{key} IS NULL)), 0) AS only_rec,
             coalesce(sum(int(r.{key} IS NOT NULL AND t.{key} IS NOT NULL AND {h('r')} <> {h('t')})), 0) AS val_diff
      FROM {view} r FULL OUTER JOIN {target} t ON r.{key} = t.{key}""").collect()[0].asDict()
    ok = r["n_rec"] == r["n_live"] and r["only_live"] == 0 and r["only_rec"] == 0 and r["val_diff"] == 0
    gate(f"{label} recompute == live", ok, r)


def hav(lon1, lat1, lon2, lat2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R_KM * np.arcsin(np.sqrt(a))


def parse_wkt(wkt: str):
    return [(float(x), float(y)) for x, y in re.findall(r"(-?[\d.]+) (-?[\d.]+)", wkt)]


def segment_route(a, z, max_km=MAX_SEG_KM, steps=ARC_STEPS):
    """BUG-QA-075: split the planar lon/lat chord a->z into n = ceil(arc_km / max_km) pieces of equal haversine arc length."""
    t = np.linspace(0, 1, steps + 1)
    lon, lat = a[0] + t * (z[0] - a[0]), a[1] + t * (z[1] - a[1])
    cum = np.concatenate([[0], np.cumsum(hav(lon[:-1], lat[:-1], lon[1:], lat[1:]))])
    n = max(1, math.ceil(cum[-1] / max_km))
    tk = np.interp(np.arange(n + 1) * cum[-1] / n, cum, t)
    tk[0], tk[-1] = 0.0, 1.0
    return n, [(a[0] + x * (z[0] - a[0]), a[1] + x * (z[1] - a[1])) for x in tk]


def slerp_densify_r6(coords, step_km=DENSIFY_KM):
    """BUG-QA-065: densify each vertex pair along the great circle (n = ceil(km / step)), coordinates rounded to 6 dp."""
    out = [coords[0]]
    for (lon1, lat1), (lon2, lat2) in zip(coords[:-1], coords[1:]):
        n = max(1, math.ceil(float(hav(lon1, lat1, lon2, lat2)) / step_km))
        if n > 1:
            v1 = np.array([math.cos(math.radians(lat1)) * math.cos(math.radians(lon1)),
                           math.cos(math.radians(lat1)) * math.sin(math.radians(lon1)), math.sin(math.radians(lat1))])
            v2 = np.array([math.cos(math.radians(lat2)) * math.cos(math.radians(lon2)),
                           math.cos(math.radians(lat2)) * math.sin(math.radians(lon2)), math.sin(math.radians(lat2))])
            om = math.acos(max(-1.0, min(1.0, float(v1 @ v2))))
            for k in range(1, n):
                f = k / n
                v = (math.sin((1 - f) * om) * v1 + math.sin(f * om) * v2) / math.sin(om)
                out.append((math.degrees(math.atan2(v[1], v[0])), math.degrees(math.asin(v[2]))))
        out.append((lon2, lat2))
    return [(round(x, 6), round(y, 6)) for x, y in out]


print(f"write_enabled={WRITE_ENABLED}  (dry run diffs only)" if not WRITE_ENABLED else "write_enabled=True: v4 candidate objects WILL be overwritten")

In [0]:
# 10,000 SPAN-* rows unchanged from span@20; 92 BB-* routes replaced by BBS-<route>-NNN segments on CDT-BBR-<route>
span_cols = spark.table(f"{S}.physical_resource_span").columns
bb_rows = [r.asDict() for r in spark.sql(
    f"SELECT * FROM {S}.physical_resource_span VERSION AS OF {PIN_SPAN} WHERE span_id LIKE 'BB-%'").collect()]

bbs = []
for r in bb_rows:
    v = parse_wkt(r["geom_wkt"])
    n, pts = segment_route(v[0], v[-1])
    segs = [(k, *pts[k - 1], *pts[k]) for k in range(1, n + 1)]
    seg_km = [float(hav(x1, y1, x2, y2)) for _, x1, y1, x2, y2 in segs]
    route_km = round(sum(seg_km), 3)
    rid = r["span_id"].split("-")[1]
    for (k, x1, y1, x2, y2), d in zip(segs, seg_km):
        bbs.append(dict(
            span_id=f"BBS-{rid}-{k:03d}", span_name=f"{r['span_name']} seg {k}/{n}", conduit_id=f"CDT-BBR-{rid}",
            site_a_id=r["site_a_id"], site_z_id=r["site_z_id"], fiber_type=r["fiber_type"], fiber_count=r["fiber_count"],
            wavelength_count=r["wavelength_count"], length_km=round(d, 3),
            geom_wkt=f"LINESTRING ({x1:.10f} {y1:.10f}, {x2:.10f} {y2:.10f})",
            install_year=r["install_year"], is_active=r["is_active"], created_at=r["created_at"],
            midpoint_lat=round((y1 + y2) / 2, 6), midpoint_lon=round((x1 + x2) / 2, 6), origin="DERIVED",
            parent_route_id=r["span_id"], segment_seq=k, segment_count=n,
            route_length_km_v3=r["length_km"], route_geodesic_km=route_km))

SPAN_V4_SCHEMA = [("span_id", "string"), ("span_name", "string"), ("conduit_id", "string"), ("site_a_id", "string"),
                  ("site_z_id", "string"), ("fiber_type", "string"), ("fiber_count", "int"), ("wavelength_count", "int"),
                  ("length_km", "double"), ("geom_wkt", "string"), ("install_year", "int"), ("is_active", "boolean"),
                  ("created_at", "timestamp"), ("midpoint_lat", "double"), ("midpoint_lon", "double"), ("origin", "string"),
                  ("parent_route_id", "string"), ("segment_seq", "int"), ("segment_count", "int"),
                  ("route_length_km_v3", "double"), ("route_geodesic_km", "double")]
bbs_df = spark.createDataFrame(pd.DataFrame(bbs)).selectExpr(*[f"CAST({c} AS {t}) AS {c}" for c, t in SPAN_V4_SCHEMA])
spans_df = spark.sql(f"""
  SELECT {', '.join(span_cols)}, CAST(NULL AS STRING) AS parent_route_id, CAST(NULL AS INT) AS segment_seq,
         CAST(NULL AS INT) AS segment_count, CAST(NULL AS DOUBLE) AS route_length_km_v3, CAST(NULL AS DOUBLE) AS route_geodesic_km
  FROM {S}.physical_resource_span VERSION AS OF {PIN_SPAN} WHERE span_id NOT LIKE 'BB-%'""")
spans_df.select(*[c for c, _ in SPAN_V4_SCHEMA]).unionByName(bbs_df).createOrReplaceTempView("v4_span")

g = spark.sql("""SELECT count(*) n, count_if(span_id LIKE 'BBS-%') n_bbs, count_if(span_id LIKE 'BB-%' AND span_id NOT LIKE 'BBS-%') n_bb,
  max(CASE WHEN span_id LIKE 'BBS-%' THEN length_km END) max_bbs_km, count(DISTINCT parent_route_id) routes FROM v4_span""").collect()[0]
gate("B1 BBS segments <= 25 km, 92 routes, 0 BB-* left", g.max_bbs_km <= MAX_SEG_KM and g.routes == 92 and g.n_bb == 0, g.asDict())

if WRITE_ENABLED:
    assert_v4_target(SPAN_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SPAN_V4} AS SELECT * FROM v4_span WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {SPAN_V4} SELECT * FROM v4_span")
diff_vs_live("v4_span", SPAN_V4, "span_id", "B1 span_v4")

In [0]:
# A1: length_km = haversine sum over geom_wkt vertices (3 dp); P35/B2 coverage from v4_span; 92 CDT-BBR-* route conduits
HAV = lambda lat1, lon1, lat2, lon2: (
    f"2*{R_KM}*ASIN(SQRT(POWER(SIN(RADIANS({lat2}-{lat1})/2),2)+COS(RADIANS({lat1}))*COS(RADIANS({lat2}))*POWER(SIN(RADIANS({lon2}-{lon1})/2),2)))")
conduit_sql = f"""
WITH v3 AS (SELECT * FROM {S}.physical_resource_conduit VERSION AS OF {PIN_CONDUIT}),
bb AS (SELECT * FROM {S}.physical_resource_span VERSION AS OF {PIN_SPAN} WHERE span_id LIKE 'BB-%'),
lines AS (SELECT conduit_id AS line_id, geom_wkt FROM v3 UNION ALL SELECT span_id, geom_wkt FROM bb),
pts AS (
  SELECT l.line_id, p.pos, CAST(split(trim(p.xy), ' ')[0] AS DOUBLE) AS lon, CAST(split(trim(p.xy), ' ')[1] AS DOUBLE) AS lat
  FROM lines l LATERAL VIEW posexplode(split(regexp_extract(l.geom_wkt, '\\\\((.*)\\\\)', 1), ',')) p AS pos, xy),
hv AS (
  SELECT line_id, pos, lon, lat,
         LAG(lon) OVER (PARTITION BY line_id ORDER BY pos) AS plon, LAG(lat) OVER (PARTITION BY line_id ORDER BY pos) AS plat
  FROM pts),
geo AS (
  SELECT line_id, ROUND(SUM(CASE WHEN plon IS NULL THEN 0 ELSE {HAV('plat', 'plon', 'lat', 'lon')} END), 3) AS geo_km,
         MIN_BY(struct(lon, lat), pos) AS f, MAX_BY(struct(lon, lat), pos) AS l
  FROM hv GROUP BY line_id),
geo2 AS (SELECT line_id, geo_km, ROUND({HAV('f.lat', 'f.lon', 'l.lat', 'l.lon')}, 3) AS chord_km FROM geo),
real_seg AS (SELECT conduit_id, ROUND(SUM(length_km), 3) AS km FROM v4_span WHERE span_id LIKE 'SPAN-%' GROUP BY 1),
bbs AS (SELECT conduit_id, ROUND(SUM(length_km), 3) AS km FROM v4_span WHERE span_id LIKE 'BBS-%' GROUP BY 1),
base AS (
  SELECT c.conduit_id, c.conduit_name, c.corridor, c.owner, c.conduit_type, c.install_year, c.site_a_id, c.site_z_id, c.geom_wkt,
         g.geo_km AS length_km, c.is_active, c.created_at, c.origin, c.length_km AS length_km_v3, g.chord_km,
         ROUND(g.geo_km - c.length_km, 3) AS length_delta_km
  FROM v3 c JOIN geo2 g ON g.line_id = c.conduit_id)
SELECT b.*,
  CASE WHEN ABS(b.length_delta_km) <= 0.0005 THEN 'UNCHANGED'
       WHEN ABS(b.length_delta_km) >= 0.01 * b.length_km THEN 'CORRECTED_GE_1PCT'
       ELSE 'CORRECTED_LT_1PCT' END AS length_change_class,
  'geodesic_sum_of_vertex_segments' AS length_method,
  CASE WHEN r.km IS NOT NULL THEN 'REAL_SEGMENTED' ELSE 'INVENTORY_ONLY' END AS span_coverage_class,
  COALESCE(r.km, 0.0) AS segmented_km,
  CASE WHEN r.km IS NULL THEN 0.0 ELSE LEAST(100.00, ROUND(100 * r.km / b.length_km, 2)) END AS coverage_pct
FROM base b LEFT JOIN real_seg r USING (conduit_id)
UNION ALL
SELECT concat('CDT-BBR-', split(s.span_id, '-')[1]),
  concat('Backbone route conduit ', s.site_a_id, '-', s.site_z_id, ' (', s.span_id, ')'),
  concat('Backbone ', s.site_a_id, '-', s.site_z_id), p.owner, p.conduit_type, s.install_year, s.site_a_id, s.site_z_id, s.geom_wkt,
  g.geo_km, s.is_active, TIMESTAMP'{B2_BUILD_TS}', 'DERIVED', CAST(NULL AS DOUBLE), g.chord_km, CAST(NULL AS DOUBLE), 'NEW_V4',
  'geodesic_sum_of_vertex_segments', 'BB_SEGMENTED', k.km, LEAST(100.00, ROUND(100 * k.km / g.geo_km, 2))
FROM bb s
JOIN v3 p ON p.conduit_id = s.conduit_id
JOIN geo2 g ON g.line_id = s.span_id
JOIN bbs k ON k.conduit_id = concat('CDT-BBR-', split(s.span_id, '-')[1])
"""
spark.sql(conduit_sql).createOrReplaceTempView("v4_conduit")

g = spark.sql("""SELECT count(*) n, count_if(span_coverage_class='REAL_SEGMENTED') real_seg, count_if(span_coverage_class='BB_SEGMENTED') bb_seg,
  count_if(span_coverage_class='INVENTORY_ONLY') inv_only FROM v4_conduit""").collect()[0]
gate("P35/B2 coverage classes 55/92/1914", (g.real_seg, g.bb_seg, g.inv_only) == (55, 92, 1914), g.asDict())
orph = spark.sql("SELECT count(*) c FROM v4_span s LEFT ANTI JOIN v4_conduit c USING (conduit_id)").collect()[0].c
gate("B2 span_v4 -> conduit_v4 FK orphans = 0", orph == 0, orph)

if WRITE_ENABLED:
    assert_v4_target(CONDUIT_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {CONDUIT_V4} AS SELECT * FROM v4_conduit WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {CONDUIT_V4} SELECT * FROM v4_conduit")
diff_vs_live("v4_conduit", CONDUIT_V4, "conduit_id", "A1/B2 conduit_v4")

In [0]:
# Rows = every v3 pair (@22, re-measured) + new in-radius pairs found by the line test. Measured on span@11 (v3 BB-* ids; BBS rebuild is BUG-QA-073).
# Distance: span line densified along great circles (5 km, 6 dp) -> closest point in EPSG:5070 -> st_distancesphere from hazard point.
sp11 = spark.sql(f"SELECT span_id, geom_wkt FROM {S}.physical_resource_span VERSION AS OF {PIN_SPAN_HSI}").collect()
dense = [(r.span_id, "LINESTRING (" + ", ".join(f"{x!r} {y!r}" for x, y in slerp_densify_r6(parse_wkt(r.geom_wkt))) + ")") for r in sp11]
spark.createDataFrame(dense, "span_id string, geom_dense_wkt string").createOrReplaceTempView("v4_hsi_span_dense")

spark.sql(f"""
  SELECT s.span_id, s.conduit_id, s.midpoint_lat, s.midpoint_lon, s.geom_wkt,
         st_transform(st_geomfromtext(d.geom_dense_wkt, 4326), 5070) AS g5070,
         st_xmin(st_geomfromtext(d.geom_dense_wkt)) AS xmin, st_xmax(st_geomfromtext(d.geom_dense_wkt)) AS xmax,
         st_ymin(st_geomfromtext(d.geom_dense_wkt)) AS ymin, st_ymax(st_geomfromtext(d.geom_dense_wkt)) AS ymax
  FROM {S}.physical_resource_span VERSION AS OF {PIN_SPAN_HSI} s JOIN v4_hsi_span_dense d USING (span_id)""").createOrReplaceTempView("v4_hsi_sp")
spark.sql(f"""
  SELECT hazard_id, hazard_type, severity, latitude, longitude, radius_km, event_start, event_end, source,
         st_transform(st_point(longitude, latitude, 4326), 5070) AS p5070,
         radius_km / 111.0 + 0.01 AS dlat, radius_km / (111.0 * greatest(cos(radians(latitude)), 0.2)) + 0.01 AS dlon
  FROM {S}.geographic_hazard_event VERSION AS OF {PIN_HAZARD}""").createOrReplaceTempView("v4_hsi_hz")

hsi_sql = f"""
WITH v3 AS (SELECT * FROM {S}.hazard_span_intersection VERSION AS OF {PIN_HSI}),
grid_pairs AS (  -- 1-degree grid buckets, then the exact bbox test (bbox = radius + 0.01 deg margin)
  SELECT DISTINCT h.hazard_id, s.span_id FROM
    (SELECT hazard_id, gx, gy FROM v4_hsi_hz
       LATERAL VIEW explode(sequence(CAST(floor(longitude - dlon) AS INT), CAST(floor(longitude + dlon) AS INT))) a AS gx
       LATERAL VIEW explode(sequence(CAST(floor(latitude - dlat) AS INT), CAST(floor(latitude + dlat) AS INT))) b AS gy) h
    JOIN (SELECT span_id, gx, gy FROM v4_hsi_sp
       LATERAL VIEW explode(sequence(CAST(floor(xmin) AS INT), CAST(floor(xmax) AS INT))) a AS gx
       LATERAL VIEW explode(sequence(CAST(floor(ymin) AS INT), CAST(floor(ymax) AS INT))) b AS gy) s
    USING (gx, gy)),
cand AS (
  SELECT h.hazard_id, s.span_id, h.radius_km,
         st_distancesphere(st_point(h.longitude, h.latitude, 4326), st_transform(st_closestpoint(s.g5070, h.p5070), 4326)) / 1000 AS line_km
  FROM grid_pairs k JOIN v4_hsi_hz h USING (hazard_id) JOIN v4_hsi_sp s USING (span_id)
  WHERE s.xmax >= h.longitude - h.dlon AND s.xmin <= h.longitude + h.dlon
    AND s.ymax >= h.latitude - h.dlat AND s.ymin <= h.latitude + h.dlat),
newp AS (
  SELECT c.hazard_id, c.span_id FROM cand c
  LEFT ANTI JOIN (SELECT DISTINCT hazard_id, span_id FROM v3) x USING (hazard_id, span_id)
  WHERE c.line_km <= c.radius_km),
span_svc AS (SELECT span_id, first(affected_service_ids) AS affected_service_ids, first(affected_service_cnt) AS affected_service_cnt
             FROM v3 GROUP BY span_id),
base AS (
  SELECT v3.intersection_id, v3.hazard_source, v3.hazard_source_id, v3.hazard_type, v3.hazard_severity,
         v3.hazard_id, v3.span_id, v3.affected_service_ids, v3.affected_service_cnt, v3.is_within_radius AS v3_is_within_radius,
         v3.distance_km AS v3_distance_km, v3.batch_id AS v3_batch_id, 'V3_PAIR' AS pair_origin
  FROM v3
  UNION ALL
  SELECT sha2(concat(h.source, h.hazard_id, n.span_id), 256), h.source, h.hazard_id, h.hazard_type, h.severity,
         n.hazard_id, n.span_id, s.affected_service_ids, s.affected_service_cnt,
         CAST(NULL AS BOOLEAN), CAST(NULL AS DOUBLE), CAST(NULL AS STRING), 'NEW_LINE_PAIR'
  FROM newp n JOIN v4_hsi_hz h USING (hazard_id) LEFT JOIN span_svc s USING (span_id)),
m AS (
  SELECT b.*, h.latitude AS hazard_lat, h.longitude AS hazard_lon, h.radius_km AS hazard_radius_km,
         h.event_start AS hazard_start, h.event_end AS hazard_end,
         sp.midpoint_lat AS span_midpoint_lat, sp.midpoint_lon AS span_midpoint_lon,
         st_transform(st_closestpoint(sp.g5070, h.p5070), 4326) AS np
  FROM base b JOIN v4_hsi_hz h USING (hazard_id) JOIN v4_hsi_sp sp USING (span_id)),
d AS (
  SELECT *, ROUND(st_distancesphere(st_point(hazard_lon, hazard_lat, 4326), np) / 1000, 4) AS distance_km,
         ROUND(st_distancesphere(st_point(hazard_lon, hazard_lat), st_point(span_midpoint_lon, span_midpoint_lat)) / 1000, 4) AS distance_to_midpoint_km
  FROM m)
SELECT intersection_id, hazard_source, hazard_source_id, hazard_type, hazard_severity, hazard_lat, hazard_lon, hazard_radius_km,
  hazard_start, hazard_end, span_id, span_midpoint_lat, span_midpoint_lon, distance_km, distance_to_midpoint_km,
  ROUND(st_y(np), 6) AS nearest_point_lat, ROUND(st_x(np), 6) AS nearest_point_lon,
  distance_km <= hazard_radius_km AS is_within_radius,
  CASE WHEN distance_km < 1 THEN 'direct_impact' WHEN distance_km < 5 THEN 'near_field' ELSE 'buffer_zone' END AS proximity_band,
  affected_service_ids, affected_service_cnt, TIMESTAMP'{HSI_BUILD_TS}' AS computed_at, '{HSI_BATCH_ID}' AS batch_id,
  hazard_id, 'DERIVED' AS origin, pair_origin, v3_is_within_radius, v3_distance_km, v3_batch_id,
  CASE WHEN pair_origin = 'NEW_LINE_PAIR' THEN 'GAINED_NEW_PAIR'
       WHEN v3_is_within_radius AND distance_km <= hazard_radius_km THEN 'UNCHANGED_IN'
       WHEN NOT v3_is_within_radius AND distance_km > hazard_radius_km THEN 'UNCHANGED_OUT'
       WHEN NOT v3_is_within_radius THEN 'GAINED_FLAG_FLIP'
       ELSE 'LOST' END AS change_vs_v3,
  'geodesic_to_linestring' AS distance_method
FROM d
"""
spark.sql(hsi_sql).createOrReplaceTempView("v4_hsi")

if WRITE_ENABLED:
    assert_v4_target(HSI_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {HSI_V4} AS SELECT * FROM v4_hsi WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {HSI_V4} SELECT * FROM v4_hsi")
diff_vs_live("v4_hsi", HSI_V4, "intersection_id", "HSI_v4")  # about 7 min on serverless

In [0]:
# BB-* route rows: rank within route by xxhash64(id), id; segment_seq = pmod(rank-1, segment_count)+1. All other rows identical to v3.
def b5_sql(src: str, id_col: str, pin: int) -> str:
    cols = spark.table(f"{S}.{src}").columns
    sel = ", ".join("g.seg_span_id AS span_id" if c == "span_id" else f"b.{c}" for c in cols)
    return f"""
    WITH seg AS (SELECT parent_route_id, segment_seq, span_id AS seg_span_id FROM v4_span WHERE span_id LIKE 'BBS-%'),
    nseg AS (SELECT parent_route_id, MAX(segment_seq) AS segment_count FROM seg GROUP BY parent_route_id),
    src AS (SELECT * FROM {S}.{src} VERSION AS OF {pin}),
    bb AS (SELECT s.*, ROW_NUMBER() OVER (PARTITION BY s.span_id ORDER BY xxhash64(s.{id_col}), s.{id_col}) AS rk
           FROM src s WHERE s.span_id LIKE 'BB-%')
    SELECT * FROM src WHERE span_id IS NULL OR span_id NOT LIKE 'BB-%'
    UNION ALL
    SELECT {sel} FROM bb b
    JOIN nseg c ON c.parent_route_id = b.span_id
    JOIN seg g ON g.parent_route_id = b.span_id AND g.segment_seq = pmod(b.rk - 1, c.segment_count) + 1"""


spark.sql(b5_sql("resource_alarm", "alarm_id", PIN_ALARM)).createOrReplaceTempView("v4_alarm")
spark.sql(b5_sql("resource_performance_measure", "measure_id", PIN_PERF)).createOrReplaceTempView("v4_perf")

for v, pin, src in (("v4_alarm", PIN_ALARM, "resource_alarm"), ("v4_perf", PIN_PERF, "resource_performance_measure")):
    g = spark.sql(f"""SELECT (SELECT count(*) FROM {v}) n_v4, (SELECT count(*) FROM {S}.{src} VERSION AS OF {pin}) n_v3,
      (SELECT count(*) FROM {v} WHERE span_id LIKE 'BB-%' AND span_id NOT LIKE 'BBS-%') n_bb_left,
      (SELECT count(*) FROM {v} a LEFT ANTI JOIN v4_span s USING (span_id) WHERE a.span_id IS NOT NULL) fk_orphans,
      (SELECT max(c) / min(c) FROM (SELECT span_id, count(*) c FROM {v} WHERE span_id LIKE 'BBS-%' GROUP BY 1)) bal""").collect()[0]
    gate(f"B5 {src}: same rows, 0 BB-* left, 0 FK orphans", g.n_v4 == g.n_v3 and g.n_bb_left == 0 and g.fk_orphans == 0, g.asDict())

if WRITE_ENABLED:
    assert_v4_target(ALARM_V4)
    assert_v4_target(PERF_V4)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ALARM_V4} AS SELECT * FROM v4_alarm WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {ALARM_V4} SELECT * FROM v4_alarm")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {PERF_V4} AS SELECT * FROM v4_perf WHERE 1 = 0")
    spark.sql(f"INSERT OVERWRITE {PERF_V4} SELECT * FROM v4_perf")
diff_vs_live("v4_alarm", ALARM_V4, "alarm_id", "B5 alarm_v4")
diff_vs_live("v4_perf", PERF_V4, "measure_id", "B5 perf_v4")

In [0]:
# Verbatim DDL (SHOW CREATE TABLE form). Dry run checks the live definition is byte-identical; write mode re-creates the view.
MV_MRC_V4_DDL = r"""CREATE VIEW cdm_tmforum.provable_resilience_intelligence.mv_mrc_protected_v4_candidate (
  Service Type COMMENT 'Service technology type (wavelength, ethernet, dark_fiber, ip_vpn, managed_security)',
  SLA Tier COMMENT 'Service SLA tier (standard, premium, premium_plus)',
  Sold as Diverse COMMENT 'Whether the service was sold with a path diversity guarantee',
  Customer Tier COMMENT 'Customer tier: platinum, gold, silver',
  Industry COMMENT 'Customer industry vertical',
  Region COMMENT 'Customer region',
  Has Completed Action COMMENT 'True if the service has at least one completed proactive_action of any type',
  MRC Protected USD COMMENT 'Business KPI. Monthly recurring charge summed over DISTINCT services with at least one completed proactive_action (any action type).',
  Services Protected COMMENT 'Distinct services with at least one completed proactive_action',
  MRC Protected Proactive Only USD COMMENT 'Supporting measure. MRC over distinct services with at least one completed reroute or preposition_splice action.',
  Services Protected Proactive Only COMMENT 'Distinct services with at least one completed reroute or preposition_splice action',
  Total MRC USD COMMENT 'Total monthly recurring charge across all services in scope (denominator)',
  MRC Protected Share COMMENT 'Share of total MRC on services with at least one completed proactive_action',
  Total Services COMMENT 'Distinct services in scope')
COMMENT 'V4 CANDIDATE (DEC-PRI-REV-PROTECTED-DEFINITION, P29-DA-REV-PROTECTED-DEDUP). Grain: one row per Lakelink Fiber service (service_id). MRC Protected = SUM of monthly recurring charge (mrc_usd) over DISTINCT services with at least one completed proactive_action. Deduplicated per service, so it never double-counts services with multiple actions. Replaces the retired \'Revenue Protected USD\' (action_queue_status) and \'revenue_protected\' (prevented_outages) at the v4 cutover. Frozen v3 metric views are unchanged.'
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

source: |-
  SELECT si.service_id, si.service_type, si.sla_tier, si.sold_as_diverse, si.mrc_usd,
         c.customer_id, c.tier AS customer_tier, c.industry, c.region,
         COALESCE(pa.completed_actions, 0) AS completed_actions,
         COALESCE(pa.total_actions, 0) AS total_actions,
         COALESCE(pa.has_completed_action, 0) AS has_completed_action,
         COALESCE(pa.has_completed_proactive, 0) AS has_completed_proactive
  FROM cdm_tmforum.provable_resilience_intelligence.service_inventory si
  LEFT JOIN cdm_tmforum.provable_resilience_intelligence.customer c ON si.customer_id = c.customer_id
  LEFT JOIN (
    SELECT service_id,
           COUNT(*) AS total_actions,
           COUNT_IF(status = 'completed') AS completed_actions,
           MAX(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS has_completed_action,
           MAX(CASE WHEN status = 'completed' AND action_type IN ('reroute','preposition_splice') THEN 1 ELSE 0 END) AS has_completed_proactive
    FROM cdm_tmforum.provable_resilience_intelligence.proactive_action
    GROUP BY service_id) pa ON si.service_id = pa.service_id

comment: "V4 CANDIDATE (DEC-PRI-REV-PROTECTED-DEFINITION, P29-DA-REV-PROTECTED-DEDUP).\
  \ Grain: one row per Lakelink Fiber service (service_id). MRC Protected = SUM of\
  \ monthly recurring charge (mrc_usd) over DISTINCT services with at least one completed\
  \ proactive_action. Deduplicated per service, so it never double-counts services\
  \ with multiple actions. Replaces the retired 'Revenue Protected USD' (action_queue_status)\
  \ and 'revenue_protected' (prevented_outages) at the v4 cutover. Frozen v3 metric\
  \ views are unchanged."

dimensions:
  - name: Service Type
    expr: service_type
    comment: "Service technology type (wavelength, ethernet, dark_fiber, ip_vpn, managed_security)"

  - name: SLA Tier
    expr: sla_tier
    comment: "Service SLA tier (standard, premium, premium_plus)"

  - name: Sold as Diverse
    expr: sold_as_diverse
    comment: Whether the service was sold with a path diversity guarantee

  - name: Customer Tier
    expr: customer_tier
    comment: "Customer tier: platinum, gold, silver"

  - name: Industry
    expr: industry
    comment: Customer industry vertical

  - name: Region
    expr: region
    comment: Customer region

  - name: Has Completed Action
    expr: has_completed_action = 1
    comment: True if the service has at least one completed proactive_action of any
      type

measures:
  - name: MRC Protected USD
    expr: SUM(mrc_usd) FILTER (WHERE has_completed_action = 1)
    comment: Business KPI. Monthly recurring charge summed over DISTINCT services
      with at least one completed proactive_action (any action type).
    display_name: MRC Protected
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - revenue protected
      - mrc protected
      - protected revenue
      - revenue saved

  - name: Services Protected
    expr: COUNT(DISTINCT service_id) FILTER (WHERE has_completed_action = 1)
    comment: Distinct services with at least one completed proactive_action

  - name: MRC Protected Proactive Only USD
    expr: SUM(mrc_usd) FILTER (WHERE has_completed_proactive = 1)
    comment: Supporting measure. MRC over distinct services with at least one completed
      reroute or preposition_splice action.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: Services Protected Proactive Only
    expr: COUNT(DISTINCT service_id) FILTER (WHERE has_completed_proactive = 1)
    comment: Distinct services with at least one completed reroute or preposition_splice
      action

  - name: Total MRC USD
    expr: SUM(mrc_usd)
    comment: Total monthly recurring charge across all services in scope (denominator)

  - name: MRC Protected Share
    expr: "SUM(mrc_usd) FILTER (WHERE has_completed_action = 1) / NULLIF(SUM(mrc_usd),\
      \ 0)"
    comment: Share of total MRC on services with at least one completed proactive_action
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1

  - name: Total Services
    expr: COUNT(DISTINCT service_id)
    comment: Distinct services in scope
$$
"""

MV_HPC_V4_DDL = r"""CREATE VIEW cdm_tmforum.provable_resilience_intelligence.hazard_proximity_count_v4_candidate (
  hazard_type COMMENT 'Type of geographic hazard (wildfire, flood, severe_weather, dig_ticket, earthquake)',
  severity COMMENT 'Hazard severity (critical, high, medium, low)',
  source COMMENT 'Data source for hazard event (noaa, usgs, 811_system, nifc)',
  is_active COMMENT 'Whether hazard is currently active (no end date) or resolved',
  event_month COMMENT 'Month when hazard event started',
  hazard_proximity_count COMMENT 'Total distinct geographic hazard events',
  active_hazard_count COMMENT 'Currently active (unresolved) hazard events',
  critical_hazard_count COMMENT 'Hazard events with critical severity',
  avg_radius_km COMMENT 'Average hazard impact radius in kilometers',
  max_radius_km COMMENT 'Maximum hazard impact radius — indicates worst-case exposure area',
  dig_ticket_count COMMENT '811 dig ticket events — excavation threats to buried fiber')
COMMENT 'V4 CANDIDATE of hazard_proximity_count for Lakelink Fiber. Excludes 1,036 future-dated (+5y shifted) wildfire events via filter event_start <= current_timestamp(); active = no end date or end in the future. Promote at v4 cutover.'
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

source: cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event

filter: event_start <= current_timestamp()

comment: "V4 CANDIDATE of hazard_proximity_count for Lakelink Fiber. Excludes 1,036\
  \ future-dated (+5y shifted) wildfire events via filter event_start <= current_timestamp();\
  \ active = no end date or end in the future. Promote at v4 cutover."

dimensions:
  - name: hazard_type
    expr: hazard_type
    comment: "Type of geographic hazard (wildfire, flood, severe_weather, dig_ticket,\
      \ earthquake)"
    display_name: Hazard Type
    synonyms:
      - threat type
      - hazard category
      - event type

  - name: severity
    expr: severity
    comment: "Hazard severity (critical, high, medium, low)"
    display_name: Severity
    synonyms:
      - hazard severity
      - threat level

  - name: source
    expr: source
    comment: "Data source for hazard event (noaa, usgs, 811_system, nifc)"
    display_name: Source System

  - name: is_active
    expr: CASE WHEN event_end IS NULL OR event_end > current_timestamp() THEN 'active'
      ELSE 'resolved' END
    comment: Whether hazard is currently active (no end date) or resolved
    display_name: Hazard Status
    synonyms:
      - active status
      - current threats

  - name: event_month
    expr: "DATE_TRUNC('MONTH', event_start)"
    comment: Month when hazard event started
    display_name: Event Month
    format:
      type: date
      date_format: locale_short_month

measures:
  - name: hazard_proximity_count
    expr: COUNT(DISTINCT hazard_id)
    comment: Total distinct geographic hazard events
    display_name: Hazard Count
    synonyms:
      - number of hazards
      - threat count
      - hazard proximity count

  - name: active_hazard_count
    expr: COUNT(DISTINCT CASE WHEN event_end IS NULL OR event_end > current_timestamp()
      THEN hazard_id END)
    comment: Currently active (unresolved) hazard events
    display_name: Active Hazards
    synonyms:
      - current threats
      - ongoing hazards

  - name: critical_hazard_count
    expr: COUNT(DISTINCT CASE WHEN severity = 'critical' THEN hazard_id END)
    comment: Hazard events with critical severity
    display_name: Critical Hazards

  - name: avg_radius_km
    expr: AVG(radius_km)
    comment: Average hazard impact radius in kilometers
    display_name: Avg Radius (km)
    format:
      type: number
      decimal_places:
        type: exact
        places: 1

  - name: max_radius_km
    expr: MAX(radius_km)
    comment: Maximum hazard impact radius — indicates worst-case exposure area
    display_name: Max Radius (km)
    format:
      type: number
      decimal_places:
        type: exact
        places: 1

  - name: dig_ticket_count
    expr: COUNT(DISTINCT CASE WHEN hazard_type = 'dig_ticket' THEN hazard_id END)
    comment: 811 dig ticket events — excavation threats to buried fiber
    display_name: Dig Ticket Count
    synonyms:
      - 811 tickets
      - excavation threats
$$
"""

for name, ddl in ((MV_MRC_V4, MV_MRC_V4_DDL), (MV_HPC_V4, MV_HPC_V4_DDL)):
    assert_v4_target(name)
    assert ddl.startswith("CREATE VIEW " + name + " ("), "DDL target mismatch for " + name
    if WRITE_ENABLED:
        spark.sql(ddl.replace("CREATE VIEW", "CREATE OR REPLACE VIEW", 1))
    live = spark.sql("SHOW CREATE TABLE " + name).collect()[0][0]  # read only
    gate(f"metric view {name.split('.')[-1]} DDL == live", live.strip() == ddl.strip(), f"{len(ddl.strip())} chars vs live {len(live.strip())}")

In [0]:
gates_pdf = pd.DataFrame(GATES)
display(gates_pdf)
failed = gates_pdf[~gates_pdf["pass"]]
assert failed.empty, f"{len(failed)} gate(s) failed: {failed['gate'].tolist()}"
print(f"All {len(gates_pdf)} gates passed (write_enabled={WRITE_ENABLED}).")